In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps

/var/folders/fb/2nwyf_mx609gfjxv_jjnb76h0000gn/T/ipykernel_12685/2128728225.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [2]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
#          host="processing-2.processing.data.production.internal",
        host="presto.processing.yoda.run",
        #host="processing.processing.data.production.internal",

        port=80,
        protocol='http',
        catalog ='hive',
        username='abhishek.akki@rapido.bike')

In [ ]:
query = """


with fact as  (
select captain_id ,city_name,captain_obj_mobile 
from 
(
    select captain_id, city_name,captain_obj_mobile,
    count(distinct case when order_status = 'dropped' then order_id end) as net_orders,
        count(distinct case when order_status = 'dropped' then yyyymmdd end) as net_days
    from 
    orders.order_logs_fact
    where yyyymmdd >= '20250201'
    and yyyymmdd <= '20250216' 
    -- and city_name = '{city}'
    -- and order_type not in ('auto','auto_premium','cab_economy','cabEconomy','cab_premium','auto_ncr','auto_lite','auto_c2c')
    and order_type in ('app')
    and city_name in ('Hyderabad','Vijayawada','Vishakapatnam','Hyderabad','Ahmedabad','Chandigarh','Kolkata')
    group by 1,2,3
    )
    where net_days >= 1

),



final_table as (
select distinct fact.captain_id,yyyymmdd,captain_obj_mobile, transaction_category,city_name
from captain.captain_transactions_snapshot
inner join fact 
on captain_transactions_snapshot.rider_id = fact.captain_id
where yyyymmdd >= '20250105'
    and yyyymmdd <= '20250203' 
and status = 'success'
and transaction_type like 'orders'
-- and service in ('Link', 'Bike Lite', 'Bike Pink', 'Bike Metro','Delivery','Magicpin Food','C2C')
and service in ('Link')
),



overall_tbl as (
select * 
from (
select *,
--row_number() over (partition by captain_id order by max_counts desc) as rank_num ,
row_number() over (partition by captain_id order by yyyymmdd desc) as rank_num_category 
from final_table 
-- order by 3,2
) 
where rank_num_category = 1 
)

select * 
from overall_tbl
where transaction_category = 'km'

"""

df = pd.read_sql(query, metabase_connection)

In [20]:
df.head()

,captain_id,yyyymmdd,captain_obj_mobile,transaction_category,city_name,rank_num_category
0,60339dd4454cef4d9c5e6160,20250201,8240093976,km,Kolkata,1
1,633b2aafe90ec63fa87da171,20250201,9123710104,km,Kolkata,1
2,5bc5432562c00e343ff804e4,20250203,7278827283,km,Kolkata,1
3,6696d96d8cad03e1fb2a65e8,20250203,7003741709,km,Kolkata,1
4,6585845be49f868fa0c7bb6b,20250126,9935768601,km,Chandigarh,1


In [21]:
df['captain_id'].nunique()

9475

In [ ]:
# df - captain_id merge left with df1 - userid 

In [23]:
df_grouped = df.groupby('city_name').agg(
    unique_captains=('captain_id', 'nunique')
).reset_index()

print(df_grouped)

       city_name  unique_captains
0      Ahmedabad               22
1     Chandigarh             1046
2      Hyderabad             3534
3        Kolkata             4460
4     Vijayawada              394
5  Vishakapatnam               19


In [34]:
df.to_csv('km_base.csv', index=False)

In [36]:
query = """

with selectors as (
select 
-- * 
distinct userid
-- distinct name, userid
--, yyyymmdd
-- selectorid, name 
-- userid, active, type, usertype, yyyymmdd, selectorid, name 
from canonical.user_selector
where 
-- createdat_utc = '20250120'
 yyyymmdd >= '20250118'
and type = 'custom'
and usertype = 'rider'
and active = true
and name
-- like ('%AHMD%')
-- city in ('Vijayawada','Hyderabad','Ahmedabad','Chandigarh','Kolkata','Vishakapatnam')

in (
'Hyd W6 Recent UHP Subs',
'Hyd W6 Recent HP Subs',
'Hyd W6 Recent MP Subs',
'Hyd W6 Recent LP Subs',
'Hyd W6 DMT UHP Subs',
'Hyd W6 DMT HP Subs',
'Hyd W6 DMT MP Subs',
'Hyd W6 DMT LP Subs',
'Hyd W6 DMT Inter Intra Subs',
'Hyd W6 Inact 1 Subs',
'Hyd W6 Inact 2 Subs',
'Hyd W6 Inact 3 Subs',
'Hyd W6 Inact 4 Subs',
'Hyd W6 Inact 5 Subs',
'Hyd W48 Subs',
'HP D HO Purchased WK 06 Kol 0 Comms',
'HP D UHO Purchased WK 06 Kol 0 Comms',
'HP Purchased WK 06 Kol 0 Comms',
'MP D UHO Purchased WK 06 Kol 0 Comms',
'MP Purchased WK 06 Kol 0 Comms',
'LP D UHO Purchased WK 06 Kol 0 Comms',
'LP Intra UHO Purchased WK 06 Kol 0 Comms',
'LP Purchased WK 06 Kol 0 Comms',
'Dormant All HP Purchased WK 06 Kol 0 Comms',
'Dormant All MP Purchased WK 06 Kol 0 Comms',
'Dormant All LP Purchased WK 06 Kol 0 Comms',
'Inact WK 06 Kol 0 Comms',
'HP D HO Non Purchased WK 06 Kol 0 Comms',
'HP D UHO Non Purchased WK 06 Kol 0 Comms',
'HP Non Purchased WK 06 Kol 0 Comms',
'MP D UHO Non Purchased WK 06 Kol 0 Comms',
'MP Non Purchased WK 06 Kol 0 Comms',
'LP D UHO Non Purchased WK 06 Kol 0 Comms',
'LP Intra UHO Non Purchased WK 06 Kol 0 Comms',
'LP Non Purchased WK 06 Kol 0 Comms',
'Dormant All HP Non Purchased WK 06 Kol 0 Comms',
'Dormant All MP Non Purchased WK 06 Kol 0 Comms',
'Dormant All LP Non Purchased WK 06 Kol 0 Comms'

)
-- limit 10
) 

select * 
from selectors

"""

df1 = pd.read_sql(query, metabase_connection)

In [37]:
df1.head()

,userid
0,5c9a24e9b50ab0456be50157
1,6306119e74c82d37de0c63f4
2,61e1250c50f34b7f728ac6ff
3,60c9cccf83f1e2f3143bccc5
4,653b4aef52c55646cad6736a


In [38]:
df1['userid'].nunique()

860685

In [39]:
df1.to_csv('subs_hyd_kol.csv', index=False)

In [ ]:
merged_df = df.merge(df1, left_on='captain_id', right_on='userid', how='left')
